# 02a_retrain_2025_12 — 2025-12 ~ 2026-04 LSTM 재학습 + final/ 자동 갱신

**작성일**: 2026-05-08  
**대상**: GPU 로컬 환경 (RTX 4090 24GB 8-way 권장)

## 진단 요약 (실행 전 반드시 읽기)

1. **`daily_panel.csv`** 가 2026-04-30 까지 forward 확장됨 ✅
2. **`target_logrv` 컬럼만** 2025-12-01 까지 finite — 5월 초 panel 빌드 당시 값으로 멈춤  
   → forward 21d 계산 갱신 안 됨
3. **fold 217~227 의 LSTM 예측이 모두 결손**:
   - fold 217~218, 221~222: max LSTM date = 2025-12-01
   - fold 219: max = 2025-10-10 (일부 결손)
   - fold 220: max = 2025-11-10 (일부 결손)
   - fold 223: 단 1일치 (12-01) 만 finite
   - fold 224~226: LSTM 0 finite (HAR-RV 만 학습됨)
4. 결과 **`ensemble_predictions_stockwise.csv` 가 2025-12-01 cutoff** → BL HOLD-OUT (2024-2025) 의 12월 weights 가 12-01 단일일 LSTM 으로 산출
5. **BL 156 pkl** 의 2025년 weights 도 잘못된 LSTM 입력에 기반 → 재계산 필요

## 본 노트북의 8 단계 (한 번 실행으로 final/ 데이터 일괄 갱신)

| 단계 | 셀 | 환경 | 시간 | 갱신 대상 |
|---|---|---|---|---|
| 1. 환경 + GPU 확인 | §1, §2 | CPU | 1분 | — |
| 2. **target_logrv 재계산** | §3 | CPU | ~3분 | `시계열_Test/.../daily_panel.csv` |
| 3. fold csv truncate (fold ≤ 216 보존) | §4 | CPU | 1분 | `시계열_Test/.../fold_predictions_stockwise.csv` |
| 4. **LSTM incremental 재학습** | §5 | **GPU** | **~40-90분** | `시계열_Test/.../fold_predictions_stockwise.csv` (fold 217+) |
| 5. ensemble 재빌드 + final/ 자동 복사 | §6 | CPU | ~5분 | **`final/phase3(data_outputs)/data/ensemble_predictions_stockwise.csv`** |
| 6. **BL 156 pkl 자동 재계산** | §7 | CPU | ~25-30분 | **`final/results/*.pkl`** (12 시점 갱신) |
| 7. 종합 검증 | §8 | CPU | 1분 | — |

**총 소요**: GPU ~40-90분 + CPU ~35-40분 = **약 1.5-2시간**

## 본 노트북 종료 후 — final/ 데이터 갱신 결과

- ✅ `final/phase3(data_outputs)/data/ensemble_predictions_stockwise.csv` (2026-04-09 까지 cover)
- ✅ `final/results/*.pkl` 156 개의 2025년 weights/ret/meta (재계산 완료)
- 다음: Streamlit 캐시 초기화 (브라우저 ⋮ → "Clear cache" → "Rerun")

## 백업 파일 (롤백용, 자동 생성)

- `시계열_Test/.../daily_panel.csv.bak_pre_retrain_2025_12`
- `시계열_Test/.../fold_predictions_stockwise.csv.bak_pre_retrain_2025_12`
- `final/phase3(data_outputs)/data/ensemble_predictions_stockwise.csv.bak_pre_retrain_2025_12`
- `final/results/_backup_pre_extension/*.pkl` (이미 존재)

## §1. 환경 + 경로 설정

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

from scripts.setup import DATA_DIR

PROJECT_ROOT = Path.cwd().parent.parent   # 시계열_Test/Phase3_Robust_Extensions/02a_*.ipynb → finance_project/
FINAL_DATA_DIR = PROJECT_ROOT / 'final' / 'phase3(data_outputs)' / 'data'

print(f'DATA_DIR        : {DATA_DIR}')
print(f'FINAL_DATA_DIR  : {FINAL_DATA_DIR}')
print(f'  (FINAL exists: {FINAL_DATA_DIR.exists()})')

## §2. GPU 환경 확인 (학습 전 반드시 검증)

In [ ]:
import torch

print(f'PyTorch     : {torch.__version__}')
print(f'CUDA avail  : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'GPU         : {torch.cuda.get_device_name(0)}')
    print(f'memory      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print(f'CUDA ver    : {torch.version.cuda}')
    print(f'\n✅ GPU 학습 준비 완료')
else:
    print('\n⚠ CUDA 미가용 — CPU 학습은 615 종목 × 12 fold × 30s = ~5시간 소요')
    print('  → CUDA + PyTorch GPU 버전 설치 후 재시도 권장')
    print('  pip uninstall torch -y && pip install torch --index-url https://download.pytorch.org/whl/cu121')

## §3. target_logrv 컬럼 재계산

**문제**: `daily_panel.csv` 의 `log_ret` 는 2026-04-30 까지 finite 하지만, `target_logrv` 만 2025-12-01 까지 finite.

**원인**: 5월 초 panel 빌드 당시 panel 끝점이 12월 초였고, 이후 panel 만 forward 확장됐지만 `target_logrv` 는 재계산되지 않음.

**해결**: 종목별 `np.log(log_ret.rolling(21).std()).shift(-21)` 재계산.  
마지막 21 거래일은 forward 데이터 부재로 NaN 이 정상 — 따라서 panel 4-30 까지면 target 4-9 까지 finite 가 정상.

In [ ]:
# §3.1 panel 로드 + target 재계산
panel_path = DATA_DIR / 'daily_panel.csv'
print(f'panel 로드 중: {panel_path}')
panel = pd.read_csv(panel_path, parse_dates=['date'])
print(f'  shape: {panel.shape}')
print(f'  date : {panel.date.min().date()} ~ {panel.date.max().date()}')

# 재계산 전 진단
tg_old = panel[np.isfinite(panel.target_logrv)]
lr_finite = panel[np.isfinite(panel.log_ret)]
print(f'\n  [재계산 전]')
print(f'  log_ret      finite max date: {lr_finite.date.max().date()}')
print(f'  target_logrv finite max date: {tg_old.date.max().date()}  ⚠')

# 재계산
print('\n  종목별 forward 21d log-RV 재계산 ...')
panel = panel.sort_values(['ticker', 'date']).reset_index(drop=True)

def _compute_target(g):
    lr = g['log_ret']
    rv_21 = lr.rolling(21).std()
    return np.log(rv_21).shift(-21)

t0 = time.time()
panel['target_logrv'] = panel.groupby('ticker', group_keys=False)['log_ret'].transform(
    lambda lr: np.log(lr.rolling(21).std()).shift(-21)
)
elapsed = time.time() - t0
print(f'  재계산 완료 ({elapsed:.1f}s)')

# 검증
tg_new = panel[np.isfinite(panel.target_logrv)]
print(f'\n  [재계산 후]')
print(f'  target_logrv finite max date: {tg_new.date.max().date()}  ✅ (예상: 4-09 부근, 4-30에서 21 영업일 전)')

In [ ]:
# §3.2 panel 저장 (백업 + 갱신)
import shutil

backup_path = panel_path.with_suffix('.csv.bak_pre_retrain_2025_12')
if not backup_path.exists():
    print(f'백업 생성: {backup_path.name}')
    shutil.copy2(panel_path, backup_path)
else:
    print(f'백업 이미 존재: {backup_path.name} (덮어쓰기 안 함)')

panel.to_csv(panel_path, index=False)
print(f'\n✅ daily_panel.csv 저장 완료: {panel_path}')
print(f'   shape: {panel.shape}')

## §4. fold_predictions_stockwise.csv 정리

**문제**: 기존 fold 217~226 의 LSTM 이 결손 (fold 217~218 = 12-01 cutoff, 219 = 10-10 cutoff, 220 = 11-10 cutoff, 223 = 12-01 단일, 224~226 = 0 finite).

**해결**: fold ≤ 216 까지만 보존 (216 의 LSTM 마지막 finite = 11-25, 정상). fold 217 부터 incremental 재학습 → 갱신된 target_logrv 로 새로 학습 + inference.

**백업**: 기존 fold csv 를 `.bak_pre_retrain_2025_12` 로 보존.

In [ ]:
# §4.1 fold csv 진단 + truncate
fold_path = DATA_DIR / 'fold_predictions_stockwise.csv'
print(f'fold 로드 중: {fold_path}')
fold = pd.read_csv(fold_path, parse_dates=['date'])
print(f'  shape: {fold.shape}')
print(f'  fold range: {fold.fold.min()} ~ {fold.fold.max()}')

# 217~226 의 LSTM finite 진단
print('\n  fold 별 LSTM finite 진단 (215 ~ 226):')
for f in range(215, fold.fold.max() + 1):
    sub = fold[fold.fold == f]
    n_lstm = np.isfinite(sub.y_pred_lstm).sum()
    if len(sub) == 0: continue
    last_lstm = sub[np.isfinite(sub.y_pred_lstm)].date.max() if n_lstm > 0 else None
    last_str = last_lstm.date() if last_lstm else 'N/A'
    pct = n_lstm / len(sub) * 100
    print(f'    fold {f}: {n_lstm}/{len(sub)} LSTM finite ({pct:.1f}%), max date = {last_str}')

# truncate: fold ≤ 216 보존 (LSTM 정상)
TRUNCATE_AT = 216
kept = fold[fold.fold <= TRUNCATE_AT].copy()
print(f'\n  fold ≤ {TRUNCATE_AT} 보존: {len(kept):,} rows ({kept.fold.nunique()} folds)')
print(f'  fold > {TRUNCATE_AT} 제거 (재학습 대상): {len(fold) - len(kept):,} rows')

In [ ]:
# §4.2 백업 + truncated 저장
import shutil

backup_fold = fold_path.with_suffix('.csv.bak_pre_retrain_2025_12')
if not backup_fold.exists():
    print(f'백업 생성: {backup_fold.name}')
    shutil.copy2(fold_path, backup_fold)
else:
    print(f'백업 이미 존재: {backup_fold.name} (덮어쓰기 안 함)')

kept.to_csv(fold_path, index=False)
print(f'\n✅ fold_predictions_stockwise.csv 갱신 완료 (fold ≤ {TRUNCATE_AT} 만 보존)')
print(f'   shape: {kept.shape}')

## §5. LSTM incremental 재학습

fold 217 부터 자동 재학습 (volatility_ensemble.run_ensemble_for_universe_parallel 의 `incremental=True`).

**예상 시간** (RTX 4090 24GB × 8-way 병렬):
- 종목당 평균 ~10-12 fold 재학습 (fold 217~227 + 추가)
- 615 종목 × 8 workers → **약 40-90분**

**중간 중단 가능**: 셀 재실행 시 incremental 모드가 이어서 학습.  
단, fold csv 의 max fold 부터 다시 시작하므로 일부 재계산.

In [ ]:
# §5.1 학습 트리거 (fold 217+ 자동 재학습)
from scripts.volatility_ensemble import run_ensemble_for_universe_parallel, V4_BEST_CONFIG

panel_path = DATA_DIR / 'daily_panel.csv'
universe_path = DATA_DIR / 'universe_full_history.csv'

assert panel_path.exists(), f'panel 없음: {panel_path}'
assert universe_path.exists(), f'universe 없음: {universe_path}'

print('=== V4_BEST_CONFIG ===')
for k, v in V4_BEST_CONFIG.items():
    print(f'  {k}: {v}')

print(f'\n⚙ 학습 시작 — incremental=True')
print(f'   기존 fold ≤ 216 보존 + fold 217+ 재학습')
print(f'   panel max date = {pd.read_csv(panel_path, usecols=["date"], parse_dates=["date"]).date.max().date()}')

t0 = time.time()
ensemble_sw = run_ensemble_for_universe_parallel(
    panel_csv=panel_path,
    universe_csv=universe_path,
    out_dir=DATA_DIR,
    config=V4_BEST_CONFIG,
    n_workers=8,                           # ⭐ RTX 4090 기준 8 권장. CPU/저메모리 GPU 시 4 또는 2 로
    out_name='ensemble_predictions_stockwise.csv',
    verbose=True,
    incremental=True,                      # ⭐ fold 216 이후 재학습
)
elapsed = time.time() - t0
print(f'\n⏱ 학습 시간: {elapsed/60:.1f} 분')
print(f'   ensemble_sw shape: {ensemble_sw.shape}')
print(f'   date 범위: {ensemble_sw.date.min().date()} ~ {ensemble_sw.date.max().date()}')

## §6. ensemble 재빌드 + final/phase3 로 복사

§5 의 출력은 `시계열_Test/Phase3_Robust_Extensions/data/ensemble_predictions_stockwise.csv` 에 저장됨.  
이를 `final/phase3(data_outputs)/data/` 로 복사하여 BL framework 가 즉시 사용할 수 있게 함.

In [ ]:
# §6.1 ensemble csv 검증
ens_src = DATA_DIR / 'ensemble_predictions_stockwise.csv'
ens = pd.read_csv(ens_src, parse_dates=['date'])
print(f'ensemble_predictions_stockwise.csv: {ens.shape}')
print(f'  date 범위: {ens.date.min().date()} ~ {ens.date.max().date()}')
print(f'  fold 범위: {ens.fold.min()} ~ {ens.fold.max()}')
print(f'  종목: {ens.ticker.nunique()}')

# 12월 ~ 4월 cover 검증
ens['ym'] = ens.date.dt.to_period('M')
month_cover = ens.groupby('ym').agg(
    n_dates=('date', 'nunique'),
    n_tickers=('ticker', 'nunique'),
)
print(f'\n  마지막 6 개월 cover:')
print(month_cover.tail(6).to_string())

# 핵심: 2025-12 가 19일 이상 cover 되어야 정상
n_dec_dates = month_cover.loc['2025-12', 'n_dates'] if pd.Period('2025-12') in month_cover.index else 0
if n_dec_dates >= 19:
    print(f'\n  ✅ 2025-12 cover {n_dec_dates} dates — 정상 (재학습 성공)')
else:
    print(f'\n  ⚠ 2025-12 cover {n_dec_dates} dates only — 재학습 미완료 또는 실패')

In [ ]:
# §6.2 final/phase3(data_outputs)/data/ 로 복사
import shutil

ens_dst = FINAL_DATA_DIR / 'ensemble_predictions_stockwise.csv'

# 백업
ens_dst_bak = ens_dst.with_suffix('.csv.bak_pre_retrain_2025_12')
if ens_dst.exists() and not ens_dst_bak.exists():
    print(f'백업 생성: {ens_dst_bak.name}')
    shutil.copy2(ens_dst, ens_dst_bak)

# 복사
FINAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(ens_src, ens_dst)
print(f'\n✅ 복사 완료: {ens_src.name} → {ens_dst}')
print(f'   size: {ens_dst.stat().st_size / 1e6:.1f} MB')

## §7. BL 156 pkl 의 12월 시점 재계산 (자동)

LSTM ensemble csv 가 갱신되었으므로 **BL pkl 의 2025년 12 시점이 새 LSTM 입력에 기반하여 재계산**되어야 합니다. 이 작업은 `final/_extend_bl_to_2025.py --recompute-2025` 가 수행하며, 본 노트북에서 자동 실행됩니다.

처리 흐름:
1. 156 BL pkl 각각 로드 → 2024-12-31 까지 truncate (기존 학습 부분 보존)
2. monthly_cache 빌드 (2010-01 ~ 2025-12, 192 시점)
3. 각 cfg 에 대해 2025-01 ~ 2025-12 (12 시점) 새로 walk_forward
   - prev_w 재구성 (2024-12 의 weights)
   - ff3_paper omega 의 op_q_prev/op_P_prev 정확 재구성
   - 갱신된 LSTM 예측 (lstm_pred_monthly) 자동 사용
4. 결과 concat 후 pkl 저장

소요: ~25-30 분 (CPU + joblib threading=4)

In [ ]:
# §7.1 BL 156 pkl 의 12월 시점 자동 재계산
import subprocess
import sys

final_dir = PROJECT_ROOT / 'final'
extend_script = final_dir / '_extend_bl_to_2025.py'

assert extend_script.exists(), f'스크립트 없음: {extend_script}'

print('=' * 70)
print(' BL 156 pkl 의 2025년 12 시점 자동 재계산 (--recompute-2025 모드)')
print('=' * 70)
print(f'
실행: python {extend_script.name} --recompute-2025')
print('예상 소요: ~25-30 분 (monthly_cache build ~10분 + 156 cfg × 12 month ~15분)')
print()

# 스크립트 실행 (final/ 폴더에서)
t0 = time.time()
proc = subprocess.run(
    [sys.executable, str(extend_script), '--recompute-2025'],
    cwd=final_dir,
    capture_output=False,   # 실시간 stdout 표시
    text=True,
)
elapsed = time.time() - t0

print()
print('=' * 70)
if proc.returncode == 0:
    print(f' ✅ BL pkl 재계산 완료 ({elapsed/60:.1f} 분)')
else:
    print(f' ❌ BL pkl 재계산 실패 (exit code {proc.returncode})')
    print(f'    {extend_script} 직접 실행 후 로그 확인')
print('=' * 70)

## §8. 종합 검증

LSTM 재학습 + ensemble 재빌드 + BL pkl 재계산 후, final/ 의 모든 데이터가 일관되었는지 종합 검증.

검증 항목:
- ensemble csv 의 2025-12 cover (19+ dates 가 정상)
- BL 156 pkl 의 ret 시리즈 길이 = 192 (2010-01 ~ 2025-12)
- mat_eq_eq_lam_pap 의 sortino_HOLD_OUT 등 12월 메트릭 변화 확인

In [ ]:
# §8.1 ensemble csv + BL pkl 종합 검증
import subprocess

final_dir = PROJECT_ROOT / 'final'
audit_script = final_dir / '_full_pkl_audit.py'

# (a) ensemble csv 의 2025-12 cover 재확인
ens_path = FINAL_DATA_DIR / 'ensemble_predictions_stockwise.csv'
print('━' * 70)
print(' (a) ensemble csv 의 2025-12 cover')
print('━' * 70)
ens = pd.read_csv(ens_path, parse_dates=['date'], usecols=['date', 'ticker'])
ens['ym'] = ens.date.dt.to_period('M')
last_6 = ens.groupby('ym').agg(n_dates=('date','nunique'), n_tickers=('ticker','nunique')).tail(6)
print(last_6.to_string())
print()
n_dec = last_6.loc[pd.Period('2025-12'), 'n_dates'] if pd.Period('2025-12') in last_6.index else 0
status = '✅ 정상' if n_dec >= 19 else f'⚠ 부족 ({n_dec} dates)'
print(f'2025-12 cover: {n_dec} dates  {status}')

# (b) BL pkl 무결성 감사
print()
print('━' * 70)
print(' (b) BL 156 pkl 무결성 감사 + 핵심 메트릭')
print('━' * 70)
proc = subprocess.run(
    [sys.executable, str(audit_script)],
    cwd=final_dir,
    capture_output=True,
    text=True,
)
# audit 결과 마지막 30 줄만 표시
print(proc.stdout[-3000:] if proc.stdout else '(no stdout)')
print()
print('=' * 70)
print(' 02a_retrain_2025_12 완료 — Streamlit 캐시 초기화 후 시연 가능')
print('=' * 70)
print()
print('다음 단계:')
print('  1. Streamlit 서버 재시작 또는 캐시 초기화 (브라우저 ⋮ → "Clear cache")')
print('  2. cd final && streamlit run app/streamlit_app.py')